In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, classification_report, f1_score

In [ ]:
with open("label_map.json", "r", encoding="utf-8") as f:
    label_map = json.load(f)

label_names = [label_map[str(i)] for i in range(13)]

In [ ]:
model = joblib.load("final_model.joblib")
X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")

In [ ]:
def analyze_model_detailed(model, X_test, y_test, label_names=None):
    y_pred = model.predict(X_test)

    # nếu không có label_names thì dùng label số
    labels = sorted(np.unique(np.concatenate([y_test, y_pred])))

    if label_names is None:
        label_names = [str(lbl) for lbl in labels]

    # report dạng dict để dễ lấy per-class F1
    report = classification_report(
        y_test,
        y_pred,
        labels=labels,
        target_names=label_names,
        output_dict=True,
        digits=4,
        zero_division=0
    )

    # confusion matrix
    cm = confusion_matrix(y_test, y_pred, labels=labels)

    # confusion matrix chuẩn hóa theo hàng
    cm_norm = confusion_matrix(y_test, y_pred, labels=labels, normalize="true")

    # bảng per-class metrics
    per_class_rows = []
    for name in label_names:
        per_class_rows.append({
            "label": name,
            "precision": report[name]["precision"],
            "recall": report[name]["recall"],
            "f1": report[name]["f1-score"],
            "support": int(report[name]["support"])
        })

    per_class_df = pd.DataFrame(per_class_rows).sort_values("f1", ascending=False)

    # metric tổng
    summary = {
        "macro_f1": f1_score(y_test, y_pred, average="macro"),
        "weighted_f1": f1_score(y_test, y_pred, average="weighted"),
        "accuracy": report["accuracy"]
    }

    return {
        "y_pred": y_pred,
        "labels": labels,
        "label_names": label_names,
        "confusion_matrix": cm,
        "confusion_matrix_norm": cm_norm,
        "per_class_df": per_class_df,
        "summary": summary,
        "report": report
    }

In [ ]:
def plot_confusion_matrix(cm, label_names, title="Confusion Matrix", fmt="d"):
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(label_names))
    plt.xticks(tick_marks, label_names, rotation=45, ha="right")
    plt.yticks(tick_marks, label_names)

    threshold = cm.max() / 2 if cm.max() > 0 else 0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            value = cm[i, j]
            text = f"{value:{fmt}}" if fmt != ".2f" else f"{value:.2f}"
            plt.text(
                j, i, text,
                horizontalalignment="center",
                color="white" if value > threshold else "black"
            )

    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.show()

In [ ]:
label_names = [str(i) for i in range(13)]

analysis = analyze_model_detailed(
    model=model,
    X_test=X_test,
    y_test=y_test,
    label_names=label_names
)

In [ ]:
print("===== SUMMARY =====")
print(f"Accuracy    : {analysis['summary']['accuracy']:.4f}")
print(f"Macro F1    : {analysis['summary']['macro_f1']:.4f}")
print(f"Weighted F1 : {analysis['summary']['weighted_f1']:.4f}")

print("\n===== PER-CLASS METRICS =====")
display(analysis["per_class_df"])

In [ ]:
# plot_confusion_matrix(
#     analysis["confusion_matrix"],
#     analysis["label_names"],
#     title="Confusion Matrix",
#     fmt="d"
# )

plot_confusion_matrix(
    analysis["confusion_matrix_norm"],
    analysis["label_names"],
    title="Normalized Confusion Matrix",
    fmt=".2f"
)

In [ ]:
# lấy class mạnh nhất và yếu nhất theo F1
per_class_df = analysis["per_class_df"]

print("===== TOP 5 CLASSES BY F1 =====")
print(per_class_df.head(5).to_string(index=False))

print("\n===== BOTTOM 5 CLASSES BY F1 =====")
print(per_class_df.tail(5).sort_values("f1", ascending=True).to_string(index=False))

In [ ]:
# các cặp nhãn hay nhầm
def most_confused_pairs(cm, label_names, top_k=10):
    pairs = []
    n = cm.shape[0]

    for i in range(n):
        for j in range(n):
            if i != j and cm[i, j] > 0:
                pairs.append({
                    "true_label": label_names[i],
                    "pred_label": label_names[j],
                    "count": int(cm[i, j])
                })

    pairs_df = pd.DataFrame(pairs).sort_values("count", ascending=False).head(top_k)
    return pairs_df

confused_df = most_confused_pairs(
    analysis["confusion_matrix"],
    analysis["label_names"],
    top_k=10
)

print("===== MOST CONFUSED LABEL PAIRS =====")
print(confused_df.to_string(index=False))